# Notebook 04 — Fabric IQ Copilot Demo

Scripted demo scenes for Copilot for Power BI, using the `University Analytics` semantic model. Each scene includes the exact Copilot prompt, a pre-run verification query, and talking points.

**Prerequisites:**
- Semantic model `University Analytics` published (see Notebook 03)
- Copilot enabled in workspace settings
- Open a Power BI report connected to the semantic model

## Pre-Demo Setup

1. Open Power BI in the Fabric portal
2. Create a new report from the `University Analytics` semantic model
3. Verify Copilot panel is available (side panel icon)
4. Run each verification query below to confirm expected data exists

## Scene 1: Enrolment Overview

**Copilot Prompt:**
> Show me a summary of student enrolments by program and semester for the last 2 academic years

**Expected output:** A matrix or bar chart showing enrolment counts by program name and semester.

In [ ]:
spark.sql("""
    SELECT p.program_name, ap.semester, ap.academic_year, COUNT(*) AS enrolment_count
    FROM university.fact_enrollments fe
    JOIN university.dim_program p ON fe.program_key = p.program_key
    JOIN university.dim_academic_period ap ON fe.academic_period_key = ap.academic_period_key
    WHERE ap.academic_year >= 2023
    GROUP BY p.program_name, ap.semester, ap.academic_year
    ORDER BY ap.academic_year, ap.semester, p.program_name
""").show(50, truncate=False)

**Talking Points — Scene 1:**
- Notice Copilot automatically joins fact and dimension tables
- Point out enrolment distribution across programs
- Highlight any semester-over-semester trends
- Show how you can refine the prompt: *"Break this down by domestic vs international"*

## Scene 2: Academic Performance Deep-Dive

**Copilot Prompt:**
> Compare the average exam scores between domestic and international students across all departments

**Expected output:** A grouped bar chart or table with average scores by department and student type.

In [ ]:
spark.sql("""
    SELECT d.department_name,
           s.domestic_international,
           ROUND(AVG(er.score_percentage), 2) AS avg_score,
           ROUND(AVG(er.grade_points), 2) AS avg_gpa,
           COUNT(*) AS exam_count
    FROM university.fact_exam_results er
    JOIN university.dim_student s ON er.student_key = s.student_key
    JOIN university.dim_course c ON er.course_key = c.course_key
    JOIN university.dim_department d ON c.department_key = d.department_key
    GROUP BY d.department_name, s.domestic_international
    ORDER BY d.department_name, s.domestic_international
""").show(20, truncate=False)

## Scene 3: Financial Health Check

**Copilot Prompt:**
> What is the outstanding balance by program, and which programs have the highest overdue amounts?

**Expected output:** A table showing total charges, payments, and outstanding balance per program.

In [ ]:
spark.sql("""
    SELECT p.program_name,
           ROUND(SUM(CASE WHEN ft.transaction_type = 'Charge' THEN ft.amount ELSE 0 END), 2) AS total_charges,
           ROUND(SUM(CASE WHEN ft.transaction_type = 'Payment' THEN ft.amount ELSE 0 END), 2) AS total_payments,
           ROUND(SUM(CASE WHEN ft.transaction_type = 'Charge' THEN ft.amount ELSE 0 END) -
                 SUM(CASE WHEN ft.transaction_type = 'Payment' THEN ft.amount ELSE 0 END), 2) AS outstanding,
           ROUND(SUM(CASE WHEN ft.is_overdue = true THEN ft.amount ELSE 0 END), 2) AS overdue_amount
    FROM university.fact_financial_transactions ft
    JOIN university.dim_student s ON ft.student_key = s.student_key
    JOIN university.dim_program p ON s.program_key = p.program_key
    GROUP BY p.program_name
    ORDER BY outstanding DESC
""").show(20, truncate=False)

## Scene 4: Student At-Risk Identification

**Copilot Prompt:**
> Show me students who have failed more than 2 courses and their current enrolment status

**Expected output:** A table listing at-risk students with their fail count and current status.

In [ ]:
spark.sql("""
    SELECT s.student_id, s.first_name, s.last_name,
           s.enrolment_status, p.program_name,
           COUNT(*) AS failed_courses
    FROM university.fact_enrollments fe
    JOIN university.dim_student s ON fe.student_key = s.student_key
    JOIN university.dim_program p ON s.program_key = p.program_key
    WHERE fe.enrollment_status = 'Failed'
    GROUP BY s.student_id, s.first_name, s.last_name, s.enrolment_status, p.program_name
    HAVING COUNT(*) > 2
    ORDER BY failed_courses DESC
""").show(20, truncate=False)

## Demo Wrap-Up

**Key takeaways to highlight:**
- Copilot understands the semantic model relationships and generates correct DAX/SQL
- Natural language queries can span multiple dimensions (student type, department, time)
- Results can be refined with follow-up prompts
- All data uses Singapore context: SGD currency, NUS 5.0 GPA scale, Aug-start academic year

**Next Steps:** Proceed to **Notebook 05** to configure a Data Agent for the staff persona.